### The University of Melbourne, School of Computing and Information Systems
# COMP90086 Computer Vision, 2025 Semester 2

## FINAL PROJECT

In [1]:
import tensorflow as tf
import pandas as pd
import os
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- Configuration ---
IMG_HEIGHT = 640
IMG_WIDTH = 480
BATCH_SIZE = 32
BASE_DIR = "Nutrition5k"
VALIDATION_SPLIT = 0.20 # 20% for validation
RANDOM_SEED = 42 # Use a seed for reproducible splits

# --- 1. Load the full labeled dataset manifest ---
full_labels_path = os.path.join(BASE_DIR, "nutrition5k_train.csv")
df = pd.read_csv(full_labels_path)
df['ID'] = df['ID'].astype(str)




In [2]:
# complete train and test split (with normalised pixel values)
df = df.assign(train_image_path=BASE_DIR + "\\train\\color\\" + df["ID"].astype(str) + "\\rgb.png",
               test_image_path=BASE_DIR + "\\test\\color\\" + df["ID"].astype(str) + "\\rgb.png")

datagen = ImageDataGenerator(rescale=1./255,
                             validation_split=VALIDATION_SPLIT)

train_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='training' # For training data
    )


validation_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='validation' # For training data
    )

Found 2641 validated image filenames.
Found 660 validated image filenames.


In [3]:
os.listdir(os.path.join(BASE_DIR, "test/color"))

['dish_3301',
 'dish_3302',
 'dish_3303',
 'dish_3304',
 'dish_3305',
 'dish_3306',
 'dish_3307',
 'dish_3308',
 'dish_3309',
 'dish_3310',
 'dish_3311',
 'dish_3312',
 'dish_3313',
 'dish_3314',
 'dish_3315',
 'dish_3316',
 'dish_3317',
 'dish_3318',
 'dish_3319',
 'dish_3320',
 'dish_3321',
 'dish_3322',
 'dish_3323',
 'dish_3324',
 'dish_3325',
 'dish_3326',
 'dish_3327',
 'dish_3328',
 'dish_3329',
 'dish_3330',
 'dish_3331',
 'dish_3332',
 'dish_3333',
 'dish_3334',
 'dish_3335',
 'dish_3336',
 'dish_3337',
 'dish_3338',
 'dish_3339',
 'dish_3340',
 'dish_3341',
 'dish_3342',
 'dish_3343',
 'dish_3344',
 'dish_3345',
 'dish_3346',
 'dish_3347',
 'dish_3348',
 'dish_3349',
 'dish_3350',
 'dish_3351',
 'dish_3352',
 'dish_3353',
 'dish_3354',
 'dish_3355',
 'dish_3356',
 'dish_3357',
 'dish_3358',
 'dish_3359',
 'dish_3360',
 'dish_3361',
 'dish_3362',
 'dish_3363',
 'dish_3364',
 'dish_3365',
 'dish_3366',
 'dish_3367',
 'dish_3368',
 'dish_3369',
 'dish_3370',
 'dish_3371',
 'dish

In [4]:
# create dataset to load test dataset
df_test = pd.DataFrame({"ID": os.listdir(os.path.join(BASE_DIR, "test/color"))})
df_test = df_test.assign(test_image_path = BASE_DIR + "\\test\\color\\" + df_test["ID"].astype(str) + "\\rgb.png")

In [5]:
df_test

,ID,test_image_path
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png
...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png


In [6]:
test_datagen=ImageDataGenerator(rescale=1./255.)
test_generator=test_datagen.flow_from_dataframe(
dataframe=df_test,
directory=os.getcwd(),
x_col="test_image_path",
y_col=None,
target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
batch_size=BATCH_SIZE,
seed=RANDOM_SEED,
shuffle=False,
class_mode=None)

Found 189 validated image filenames.


In [ ]:
# build basic regression model
model = tf.keras.Sequential(
    [
        layers.Input((IMG_HEIGHT, IMG_WIDTH, 3)),
        
        layers.Conv2D(8, (3, 3), activation='relu'), # fill in
        layers.MaxPooling2D((2, 2)), # fill in
        
        layers.Flatten(),
        layers.Dense(10, activation="relu"),
        layers.Dense(1, activation="linear")
    ], 
)

model.compile(optimizer='adam', loss='mse', metrics=['mse'])

ValueError: Kernel shape must have the same length as input, but received kernel of shape (3, 3, 3, 8) and input of shape (None, 32, 640, 480, 3).

In [ ]:
STEP_SIZE_TRAIN = train_generator.n//train_generator.batch_size
STEP_SIZE_VALID = validation_generator.n//validation_generator.batch_size

model.fit(train_generator,
          steps_per_epoch=STEP_SIZE_TRAIN,
          validation_data=validation_generator,
          validation_steps=STEP_SIZE_VALID,
          epochs=20
)


Epoch 1/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 622ms/step - loss: 16333.6514 - mse: 16333.6514 - val_loss: 18809.3867 - val_mse: 18809.3867
Epoch 2/20
 1/82 ━━━━━━━━━━━━━━━━━━━━ 21s 261ms/step - loss: 429735.1875 - mse: 429735.1875

c:\Users\nares\anaconda3\envs\CV\lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


82/82 ━━━━━━━━━━━━━━━━━━━━ 8s 98ms/step - loss: 429735.1875 - mse: 429735.1875 - val_loss: 19897.7402 - val_mse: 19897.7402
Epoch 3/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 50s 607ms/step - loss: 28186.5703 - mse: 28186.5703 - val_loss: 20509.4023 - val_mse: 20509.4023
Epoch 4/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 88ms/step - loss: 12755.5283 - mse: 12755.5283 - val_loss: 19218.4492 - val_mse: 19218.4492
Epoch 5/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 48s 589ms/step - loss: 15033.2031 - mse: 15033.2031 - val_loss: 16551.2500 - val_mse: 16551.2500
Epoch 6/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 8s 90ms/step - loss: 11581.6562 - mse: 11581.6562 - val_loss: 17622.5977 - val_mse: 17622.5977
Epoch 7/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 46s 559ms/step - loss: 13969.6758 - mse: 13969.6758 - val_loss: 15757.3770 - val_mse: 15757.3770
Epoch 8/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 88ms/step - loss: 4708.4004 - mse: 4708.4004 - val_loss: 15937.9160 - val_mse: 15937.9160
Epoch 9/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 46s 561ms/step - loss: 8377.0410 - m

In [ ]:
# validation
STEP_SIZE_TEST=test_generator.n//test_generator.batch_size
model.evaluate(validation_generator)

21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 344ms/step - loss: 14278.6387 - mse: 14278.6387


[14662.9150390625, 14662.9150390625]

In [ ]:
test_generator.reset()
preds=model.predict(test_generator)

6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 376ms/step


In [ ]:
df_test = df_test.assign(Value=preds)

In [ ]:
df_test

,ID,test_image_path,Value
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png,853.766052
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png,152.479446
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png,91.166473
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png,161.127014
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png,360.575531
...,...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png,132.760986
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png,0.033012
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png,378.267456
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png,165.710480


In [ ]:
df_submit = df_test.drop("test_image_path", axis=1)

In [ ]:
df_submit

,ID,Value
0,dish_3301,853.766052
1,dish_3302,152.479446
2,dish_3303,91.166473
3,dish_3304,161.127014
4,dish_3305,360.575531
...,...,...
184,dish_3485,132.760986
185,dish_3486,0.033012
186,dish_3487,378.267456
187,dish_3488,165.710480


In [ ]:
# save copy to csv
df_submit.to_csv("iteration_baseline_submission.csv", index=False)